# Testing-Effect Question Generation (QG) — Training (pilot)

Trains `t5-small` on the (input_ids, attention_mask, labels) produced by
`08_qg_testing_effect_prep.ipynb`. Pilot (`-small`); once validated, scale
up to `t5-base`.

Metric: **ROUGE-L** against the reference question (standard for
generation tasks evaluated against a single reference string). Unlike A's
novel n-gram manipulation check (rehearsal must stay verbatim), QG's job is
specifically to produce *new* text — a question, not a copy of anything in
the input — so there's no equivalent "stay close to the source" check here.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import evaluate
import numpy as np
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data/model

Loads the tokenized dataset (train 3,000-ish / val 300-ish / test 300-ish,
produced by `08_qg_testing_effect_prep.ipynb`) and subsamples to a pilot
size. `PILOT_TRAIN_SIZE`/`PILOT_VAL_SIZE`/`PILOT_TEST_SIZE` exist purely to
validate the code path — this is not a quality benchmark.

`val` drives model selection during training (`load_best_model_at_end`/
`metric_for_best_model="rougeL"`) — `test` is never touched until §5, after
training is completely finished.

In [2]:
MODEL_NAME = "t5-small"
DATA_DIR = Path("data/processed/qg_testing_effect")
OUTPUT_DIR = "experiments/qg_testing_effect_small"
PILOT_TRAIN_SIZE = 200
PILOT_VAL_SIZE = 20
PILOT_TEST_SIZE = 20

DATA_READY = all((DATA_DIR / split).exists() for split in ("train", "val", "test"))
if not DATA_READY:
    print(f"No tokenized data at {DATA_DIR} — run 08_qg_testing_effect_prep.ipynb first.")
else:
    full_train_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    full_val_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))
    full_test_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "test"))

    train_dataset = full_train_dataset.select(range(min(PILOT_TRAIN_SIZE, len(full_train_dataset))))
    val_dataset = full_val_dataset.select(range(min(PILOT_VAL_SIZE, len(full_val_dataset))))
    test_dataset = full_test_dataset.select(range(min(PILOT_TEST_SIZE, len(full_test_dataset))))
    print(f"Using {len(train_dataset)}/{len(full_train_dataset)} train rows, "
          f"{len(val_dataset)}/{len(full_val_dataset)} val rows, "
          f"{len(test_dataset)}/{len(full_test_dataset)} test rows")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    print(f"{MODEL_NAME} loaded, parameter count: {sum(p.numel() for p in model.parameters()):,}")


Using 200/3000 train rows, 20/300 val rows, 20/300 test rows


t5-small loaded, parameter count: 60,506,624


## 2. Metrics

ROUGE-L between generated and reference question.

In [3]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {"rougeL": result["rougeL"]}

## 3. Train

`Seq2SeqTrainer` + `Seq2SeqTrainingArguments`, same pattern as
`05_rehearsal_maintenance_train.ipynb`. Checkpoints go to
`experiments/qg_testing_effect_small/`.

In [4]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=32,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

  0%|          | 0/50 [00:00<?, ?it/s]

/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


  2%|▏         | 1/50 [00:05<04:23,  5.37s/it]

  4%|▍         | 2/50 [00:07<02:33,  3.20s/it]

...

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


100%|██████████| 50/50 [01:33<00:00,  1.05s/it]

100%|██████████| 50/50 [01:33<00:00,  1.87s/it]

{'train_runtime': 93.4934, 'train_samples_per_second': 2.139, 'train_steps_per_second': 0.535, 'train_loss': 3.346625747680664, 'epoch': 1.0}


TrainOutput(global_step=50, training_loss=3.346625747680664, metrics={'train_runtime': 93.4934, 'train_samples_per_second': 2.139, 'train_steps_per_second': 0.535, 'total_flos': 15162511196160.0, 'train_loss': 3.346625747680664, 'epoch': 1.0})

## 4. Save final model

In [5]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

Saved to: experiments/qg_testing_effect_small


## 5. Final evaluation — held-out test set

`test_dataset` was never used during training (not for gradient updates, not
for checkpoint selection) — this is the one point where it gets touched.
`eval_rougeL` from the training log above is a validation-set number used to
pick the best checkpoint; reporting that as the final result would be
circular.

In [6]:
if not DATA_READY:
    print("No data — skipping final test evaluation.")
else:
    test_trainer = Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_eval_batch_size=4,
            predict_with_generate=True,
            generation_max_length=32,
            report_to="none",
        ),
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    test_metrics = test_trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
    print("Final test-set metrics:", test_metrics)


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


  0%|          | 0/5 [00:00<?, ?it/s]

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


 40%|████      | 2/5 [00:01<00:01,  1.56it/s]

...

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


 80%|████████  | 4/5 [00:03<00:00,  1.15it/s]

100%|██████████| 5/5 [00:04<00:00,  1.07it/s]

100%|██████████| 5/5 [00:04<00:00,  1.15it/s]

Final test-set metrics: {'test_loss': 2.760651111602783, 'test_model_preparation_time': 0.0003, 'test_rougeL': 0.2320495056991112, 'test_runtime': 5.7838, 'test_samples_per_second': 3.458, 'test_steps_per_second': 0.864}


## 6. Qualitative check — generate a question from a real chunk

Uses the same held-out `cnn_dailymail` test-split article that
`05_rehearsal_maintenance_train.ipynb` validated rehearsal on (Palestinian
Authority / ICC), picking one concrete fact from it as the target answer —
this is the actual testing-effect use case: given a chunk and a fact within
it, generate a quiz question. This is a spot-check on one human-readable
example; §5 above is the actual quantitative test-set result.

In [7]:
if not DATA_READY:
    print("No data — skipping validation.")
else:
    from datasets import load_dataset

    device = next(model.parameters()).device
    test_article = load_dataset("cnn_dailymail", "3.0.0", split="test[0:1]")[0]["article"]
    chunk_text = " ".join(test_article.split()[:350])
    answer = "123rd"

    input_text = f"answer: {answer} context: {chunk_text}"
    input_ids = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(device)
    output_ids = model.generate(input_ids, max_new_tokens=32)
    generated_question = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"answer (target fact): {answer!r}")
    print(f"generated question: {generated_question!r}")
    print(f"\nchunk (first 300 chars): {chunk_text[:300]}")


answer (target fact): '123rd'
generated question: 'The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday.'

chunk (first 300 chars): (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou


## Summary

- This pilot: `PILOT_TRAIN_SIZE`/`PILOT_VAL_SIZE`/`PILOT_TEST_SIZE` = 200/20/20
  of `squad`, `t5-small`, 1 epoch (50 steps). Train loss 3.87 → 2.98,
  validation `eval_rougeL` 0.079, **final held-out test_rougeL 0.232** — the
  test score being higher than validation here is very likely just noise
  from a 20-example test set, not a real signal; don't read anything into
  the direction of that gap until `PILOT_TEST_SIZE` is scaled up.
- `test` (held out from training and model selection entirely) is only
  touched once, in §5, for the metric that should actually be reported.
- As already noted in the qualitative check (§6): at this pilot scale the
  model hasn't learned to actually *ask* a question yet — it echoes a
  declarative sentence back instead of an interrogative. The reasonably
  competitive test_rougeL despite that is a reminder that ROUGE-L alone is a
  weak proxy for "is this a good quiz question."
- Next: (1) scale up `PILOT_TRAIN_SIZE`/`PILOT_TEST_SIZE` before drawing any
  quality conclusions, (2) swap backbone to `t5-base`, (3) qualitatively
  review a batch of generated questions for whether the answer span is
  actually recoverable from them, (4) record results in Obsidian.